Basic Setting

In [1]:
import duckdb
import anthropic
import re
import os

In [ ]:
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
con = duckdb.connect('/Users/isabella/data/fraud_analytics.db')

In [4]:
with open('./schema_docs/schema.md', 'r') as f:
    schema_context = f.read()

In [5]:
def run_query(sql: str) -> str:
    try:
        result = con.execute(sql).df()
        if result.empty:
            return "Query returned no results."
        #Limit returned length of rows, avoid too long context
        if len(result) > 20:
            return result.head(20).to_string(index=False) + f"\n... ({len(result)} rows total, showing top 20)"
        return result.to_string(index=False)
    except Exception as e:
        return f"SQL ERROR: {str(e)}"

Agent with Memory

In [6]:
class FraudAnalyticsAgent:
    """
    Text-to-SQL agent with conversation memory
    Maintain the full conversational history and support multi-turn context
    """
    
    def __init__(self):
        self.conversation_history = []
        self.last_sql = None
        self.last_result = None
        self.turn_count = 0
        
        self.system_prompt = f"""You are a data analyst for a banking fraud analytics platform.
You help users explore a 13M-row financial transactions database through natural language.

{schema_context}

Rules:
- Always wrap SQL in ```sql ... ``` blocks
- amount can be negative (refunds) — filter amount > 0 for spend analysis  
- For fraud rate: use AVG(is_fraud)*100, not SUM
- Always INNER JOIN fraud_labels (never LEFT JOIN)
- Limit results to 20 rows unless user asks for more
- You have memory of the full conversation — use it for follow-up questions

When the user asks a follow-up like "just show top 3", "break that down by X", 
"now filter to Y", or "same but for Z" — reference the previous query context
and modify accordingly. Always show what changed.
"""
    
    def chat(self, user_message: str) -> str:
        self.turn_count += 1
        print(f"\n{'='*55}")
        print(f"Turn {self.turn_count} — User: {user_message}")
        print('='*55)
        
        #If have last result, add it to context
        if self.last_sql and self.last_result:
            context_note = f"\n\n[Previous query for reference:\n```sql\n{self.last_sql}\n```\nResult preview: {self.last_result[:300]}]"
            full_message = user_message + context_note
        else:
            full_message = user_message
        
        #Add conversation history
        self.conversation_history.append({
            "role": "user",
            "content": full_message
        })
        
        # Step 1: Generate SQL
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=1024,
            system=self.system_prompt,
            messages=self.conversation_history
        )
        
        agent_reply = response.content[0].text
        
        # Extract SQL
        sql_match = re.search(r'```sql\s*(.*?)\s*```', agent_reply, re.DOTALL)
        
        if not sql_match:
            # If no SQL match，could be clarifying questions or can't answer
            self.conversation_history.append({
                "role": "assistant",
                "content": agent_reply
            })
            print(f"Agent: {agent_reply}")
            return agent_reply
        
        sql = sql_match.group(1).strip()
        print(f"\nSQL:\n{sql}")
        
        # Step 2: Execute SQL
        query_result = run_query(sql)
        print(f"\nResult:\n{query_result[:400]}")
        
        # Step 3: Generate insight
        result_message = f"Query executed. Results:\n{query_result}\n\nPlease summarize the findings in 2-3 sentences, and if this is a follow-up question, highlight what changed compared to the previous result."
        
        self.conversation_history.append({
            "role": "assistant", 
            "content": agent_reply
        })
        self.conversation_history.append({
            "role": "user",
            "content": result_message
        })
        
        interpretation = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=512,
            system=self.system_prompt,
            messages=self.conversation_history
        )
        
        final_answer = interpretation.content[0].text
        
        # Update history and memory
        self.conversation_history.append({
            "role": "assistant",
            "content": final_answer
        })
        self.last_sql = sql
        self.last_result = query_result
        
        print(f"\nInsight: {final_answer}")
        return final_answer
    
    def reset(self):
        """Start new conversation"""
        self.conversation_history = []
        self.last_sql = None
        self.last_result = None
        self.turn_count = 0
        print("✓ Conversation reset.")
    
    def show_history(self):
        """Show summary of history conversation"""
        print(f"\nConversation history: {len(self.conversation_history)} messages, {self.turn_count} turns")

Test multi turn conversations

In [7]:
agent = FraudAnalyticsAgent()

# Round One：Basic Question
agent.chat("Which states have the highest fraud rates?")


Turn 1 — User: Which states have the highest fraud rates?

SQL:
SELECT 
    t.merchant_state,
    COUNT(*) AS total_txns,
    SUM(fl.is_fraud) AS fraud_count,
    AVG(fl.is_fraud) * 100 AS fraud_rate_pct
FROM transactions t
JOIN fraud_labels fl ON t.id = fl.transaction_id
WHERE t.merchant_state IS NOT NULL
GROUP BY t.merchant_state
ORDER BY fraud_rate_pct DESC
LIMIT 20

Result:
merchant_state  total_txns  fraud_count  fraud_rate_pct
        Tuvalu           5          5.0      100.000000
         Haiti         264        253.0       95.833333
         Italy        4706       3061.0       65.044624
            OH      324098        316.0        0.097501
            SD       20875          8.0        0.038323
            IA      107765         32.0        0.029694
        

Insight: **Key Findings:**

The data reveals extreme fraud rates in international locations — **Tuvalu (100%), Haiti (95.8%), and Italy (65%)** — though these have relatively low transaction volumes (5, 264, and 4,70

'**Key Findings:**\n\nThe data reveals extreme fraud rates in international locations — **Tuvalu (100%), Haiti (95.8%), and Italy (65%)** — though these have relatively low transaction volumes (5, 264, and 4,706 respectively). Among U.S. states, fraud rates are dramatically lower and more consistent, with **Ohio leading at 0.098%**, followed by South Dakota (0.038%) and Iowa (0.030%). The stark contrast suggests either international transactions face different risk profiles, or these may represent data quality issues worth investigating further.'

In [8]:
# Round Two：follow-up，narrow down
agent.chat("Just show me the top 3, and add the total number of fraudulent transactions for each")


Turn 2 — User: Just show me the top 3, and add the total number of fraudulent transactions for each

SQL:
SELECT 
    t.merchant_state,
    COUNT(*) AS total_txns,
    SUM(fl.is_fraud) AS fraud_count,
    AVG(fl.is_fraud) * 100 AS fraud_rate_pct
FROM transactions t
JOIN fraud_labels fl ON t.id = fl.transaction_id
WHERE t.merchant_state IS NOT NULL
GROUP BY t.merchant_state
ORDER BY fraud_rate_pct DESC
LIMIT 3

Result:
merchant_state  total_txns  fraud_count  fraud_rate_pct
        Tuvalu           5          5.0      100.000000
         Haiti         264        253.0       95.833333
         Italy        4706       3061.0       65.044624

Insight: **Key Findings:**

The top 3 highest fraud rate locations are all international: **Tuvalu (100% - all 5 transactions fraudulent), Haiti (95.8% - 253 out of 264 transactions), and Italy (65% - 3,061 out of 4,706 transactions)**. These results confirm that international transactions represent extreme fraud risk compared to domestic U.S. states

'**Key Findings:**\n\nThe top 3 highest fraud rate locations are all international: **Tuvalu (100% - all 5 transactions fraudulent), Haiti (95.8% - 253 out of 264 transactions), and Italy (65% - 3,061 out of 4,706 transactions)**. These results confirm that international transactions represent extreme fraud risk compared to domestic U.S. states, with Italy showing particularly concerning volume at over 3,000 fraudulent transactions.\n\n**What changed:** Narrowed from 20 states down to just the top 3 as requested. The fraud_count column was already present in the previous query - no changes to the data retrieved, just fewer rows displayed.'

In [9]:
# Round Three：pivot analysis
agent.chat("For those top 3 states, break it down by card type")


Turn 3 — User: For those top 3 states, break it down by card type

SQL:
SELECT 
    t.merchant_state,
    c.card_type,
    COUNT(*) AS total_txns,
    SUM(fl.is_fraud) AS fraud_count,
    AVG(fl.is_fraud) * 100 AS fraud_rate_pct
FROM transactions t
JOIN fraud_labels fl ON t.id = fl.transaction_id
JOIN cards c ON t.card_id = c.id
WHERE t.merchant_state IN ('Tuvalu', 'Haiti', 'Italy')
GROUP BY t.merchant_state, c.card_type
ORDER BY t.merchant_state, fraud_rate_pct DESC

Result:
merchant_state       card_type  total_txns  fraud_count  fraud_rate_pct
         Haiti          Credit          91         89.0       97.802198
         Haiti Debit (Prepaid)          27         26.0       96.296296
         Haiti           Debit         146        138.0       94.520548
         Italy Debit (Prepaid)         306        244.0       79.738562
         Italy          Credit        16

Insight: **Key Findings:**

Across all three high-risk states, **every card type shows extremely high fraud rates** 

'**Key Findings:**\n\nAcross all three high-risk states, **every card type shows extremely high fraud rates** - even the "best" category (Italy Debit at 59%) would be catastrophic in normal operations. In **Haiti, Credit cards have the highest fraud rate at 97.8%**, while in **Italy, Debit (Prepaid) cards lead at 79.7%**. Tuvalu shows 100% fraud across both Credit and Debit, though with minimal volume (2 and 3 transactions respectively).\n\n**What changed:** Added card type breakdown by joining the cards table and grouping by both merchant_state and card_type. Instead of 3 rows (one per state), we now see 8 rows showing how fraud rates vary by payment method within each of these high-risk international locations.'

In [10]:
# Round Four：completely different direction，test if memory ensures no confusion
agent.chat("Now show me the fraud rate by hour of day instead")


Turn 4 — User: Now show me the fraud rate by hour of day instead

SQL:
SELECT 
    EXTRACT(HOUR FROM CAST(t.date AS TIMESTAMP)) AS hour_of_day,
    COUNT(*) AS total_txns,
    SUM(fl.is_fraud) AS fraud_count,
    AVG(fl.is_fraud) * 100 AS fraud_rate_pct
FROM transactions t
JOIN fraud_labels fl ON t.id = fl.transaction_id
GROUP BY hour_of_day
ORDER BY hour_of_day

Result:
 hour_of_day  total_txns  fraud_count  fraud_rate_pct
           0       94170         13.0        0.013805
           1       77322         38.0        0.049145
           2       75384         38.0        0.050409
           3       69297        107.0        0.154408
           4       77190        143.0        0.185257
           5      122510        242.0        0.197535
           6      5080

Insight: **Key Findings:**

Fraud rates show a clear temporal pattern with **peak fraud occurring mid-day between 10 AM-1 PM (0.25-0.26%)** - roughly double the baseline rate. There's a notable **early morning spike at 3-5 

"**Key Findings:**\n\nFraud rates show a clear temporal pattern with **peak fraud occurring mid-day between 10 AM-1 PM (0.25-0.26%)** - roughly double the baseline rate. There's a notable **early morning spike at 3-5 AM (0.15-0.20%)** when transaction volumes are lowest, and fraud rates drop significantly during **evening/night hours (6-11 PM at ~0.14-0.16%)** and reach their lowest overnight at **midnight (0.014%)**.\n\n**What changed:** Completely shifted from analyzing geographic/card type patterns in high-risk international locations to examining temporal fraud patterns across all transactions globally. Instead of 8 rows showing state/card breakdowns, we now have 24 rows (one per hour) revealing when fraud is most likely to occur throughout the day."

In [11]:
# Round Five：follow-up again
agent.chat("Which of those hours are considered late night? (10pm to 4am)")


Turn 5 — User: Which of those hours are considered late night? (10pm to 4am)

SQL:
SELECT 
    EXTRACT(HOUR FROM CAST(t.date AS TIMESTAMP)) AS hour_of_day,
    COUNT(*) AS total_txns,
    SUM(fl.is_fraud) AS fraud_count,
    AVG(fl.is_fraud) * 100 AS fraud_rate_pct
FROM transactions t
JOIN fraud_labels fl ON t.id = fl.transaction_id
GROUP BY hour_of_day
HAVING hour_of_day IN (22, 23, 0, 1, 2, 3, 4)
ORDER BY hour_of_day

Result:
 hour_of_day  total_txns  fraud_count  fraud_rate_pct
           0       94170         13.0        0.013805
           1       77322         38.0        0.049145
           2       75384         38.0        0.050409
           3       69297        107.0        0.154408
           4       77190        143.0        0.185257
          22      280562         13.0        0.004634
          23      1063

Insight: **Key Findings:**

Late night hours (10pm-4am) show **dramatically varying fraud patterns**: the period starts very safe at **10-11pm (0.005-0.02%)**, drops

'**Key Findings:**\n\nLate night hours (10pm-4am) show **dramatically varying fraud patterns**: the period starts very safe at **10-11pm (0.005-0.02%)**, drops to the day\'s absolute lowest at **midnight (0.014%)**, then **surges steadily from 1-4am, peaking at 4am (0.19%)** - nearly 40x higher than 10pm. This suggests fraudsters become more active in the pre-dawn hours (3-4am) when legitimate transaction volume is lowest and monitoring may be reduced.\n\n**What changed:** Filtered the 24-hour dataset down to just the 7 late-night hours (10pm-4am) as you defined. This reveals that "late night" isn\'t uniformly risky - there\'s a U-shaped pattern with very low fraud at the start of the night (10pm-midnight) and significantly elevated fraud in the early morning hours (3-4am).'